# Classificação do BrainLat com os modelos treinados no ds004504

Fica na raiz do projeto, ao lado de `preprocessingData.ipynb` e `models_training.ipynb`.

1. Monta a coorte (AD e CN) a partir de `brainlat_eeg/` e dos CSVs de cognição.
2. Confere a adaptação de canais num sujeito de cada sítio (AR e CL).
3. Converte cada registro para o espaço do treino: 19 canais, 500 Hz, na mesma referência-alvo
   do ds004504 (padrão: média dos 19 canais, independente da referência de gravação).
4. Roda o mesmo pré-processamento e as mesmas features do `preprocessingData`.
5. Treina e salva os modelos finais (se ainda não existirem) e classifica.

## 0 - caminhos

In [ ]:
import sys, importlib, json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import mne
import matplotlib.pyplot as plt

# Raiz do projeto = pasta que contém brainlat_eeg/ (sobe a partir da pasta atual)
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "brainlat_eeg").is_dir()), None)
assert PROJECT_ROOT is not None, f"não achei a pasta brainlat_eeg/ a partir de {Path.cwd()}"
sys.path.insert(0, str(PROJECT_ROOT))
import feature_sets, modeling, channel_adaptation, ds004504_features, brainlat_transfer, train_final_models
for m in (feature_sets, modeling, channel_adaptation, ds004504_features, brainlat_transfer, train_final_models):
    importlib.reload(m)

BRAINLAT_ROOT = PROJECT_ROOT / "brainlat_eeg"

# Procura os CSVs de rótulos em qualquer lugar do projeto (aceita nomes como
# "Cognition_AD_EEG_data (1).csv"). Se preferir, escreva os caminhos à mão:
# LABEL_CSVS = [Path(".../Cognition_AD_EEG_data.csv"), Path(".../Cognition_HC_EEG_data.csv")]
LABEL_CSVS = []
for stem in ("Cognition_AD_EEG_data", "Cognition_HC_EEG_data"):
    hits = sorted(PROJECT_ROOT.rglob(f"{stem}*.csv"), key=lambda p: (len(p.parts), str(p)))
    if not hits:
        raise FileNotFoundError(
            f"{stem}.csv não foi encontrado em nenhuma subpasta de {PROJECT_ROOT}. "
            "Copie os dois CSVs de cognição para essa pasta (ou qualquer subpasta dela).")
    if len(hits) > 1:
        print(f"{stem}: {len(hits)} cópias, usando a mais próxima da raiz:",
              [str(h.relative_to(PROJECT_ROOT)) for h in hits])
    LABEL_CSVS.append(hits[0])

TRAIN_FEATURES = PROJECT_ROOT / "eeg_features_ds004504.csv"
MODEL_BUNDLE   = PROJECT_ROOT / "models" / "ds004504_models.joblib"
FEATURE_VERSION = feature_sets.FEATURE_VERSION
FEATURE_TAG    = feature_sets.feature_tag()
CACHE_DIR      = PROJECT_ROOT / "brainlat_features" / FEATURE_TAG   # um CSV + um JSON por sujeito
OUT_DIR        = PROJECT_ROOT / "resultados_brainlat"
OUT_DIR.mkdir(exist_ok=True)

# Opcional, mas recomendado: lê a ordem real dos canais do treino
TRAINING_SET_EXAMPLE = PROJECT_ROOT / "88 subjects" / "sub-001" / "eeg" / "sub-001_task-eyesclosed_eeg.set"
ORDER = channel_adaptation.training_channel_order(TRAINING_SET_EXAMPLE)

print("mne         :", mne.__version__)
print("features    :", FEATURE_TAG, "| cache:", CACHE_DIR.relative_to(PROJECT_ROOT))
print("projeto     :", PROJECT_ROOT)
print("rótulos     :", [str(p.relative_to(PROJECT_ROOT)) for p in LABEL_CSVS])
print("features    :", TRAIN_FEATURES.name, "OK" if TRAIN_FEATURES.exists() else "AUSENTE")
print("ordem treino:", ORDER)

## 1 - coorte

Para cada linha dos CSVs, procura `brainlat_eeg/*/<sítio>/<id>` e o arquivo EEG dentro da
pasta. Também confere se a pasta de grupo (`1_AD`, `5_HC`) bate com o diagnóstico do CSV.

In [ ]:
cohort = brainlat_transfer.build_cohort(BRAINLAT_ROOT, LABEL_CSVS, positive=("AD",), negative=("CN",))
print(cohort.groupby(["diagnosis", "site", "status"]).size().to_string())

problems = cohort[cohort["status"] != "ok"]
if len(problems):
    print("\nexcluídos:")
    print(problems[["sid", "diagnosis", "site", "status"]].to_string(index=False))
multi = cohort[cohort["n_eeg_files"] > 1]
if len(multi):
    print(f"\n{len(multi)} sujeitos com mais de um arquivo EEG; arquivo escolhido:")
    for r in multi.itertuples():
        print(f"  {r.sid}: {Path(r.eeg_file).name}  (de {r.n_eeg_files})")
print("\npastas sem rótulo nos CSVs:", cohort.attrs["unlabeled_on_disk"] or "nenhuma")

ok = cohort[cohort["status"] == "ok"].copy()
paths  = dict(zip(ok["sid"], ok["eeg_file"]))
labels = pd.Series(ok["y"].values, index=ok["sid"], name="y")
sites  = pd.Series(ok["site"].values, index=ok["sid"], name="site")
print(f"\ncoorte final: {len(ok)} ({labels.sum()} AD, {(labels == 0).sum()} CN)")

## 2 - estado dos arquivos e adaptação de canais: um sujeito de cada sítio

Esta é a etapa mais importante. Para cada sítio, confira primeiro o **estado do arquivo**:
- canais fora do escalpo (ECG, EXG): o nome `rs-HEP` sugere que há ECG;
- filtros já aplicados: o treino usa 0,5-45 Hz, e um passa-baixa abaixo de 45 Hz afeta a banda gama;
- referência média, trechos removidos (`boundary`) e amplitude.

Depois, a **adaptação**:
- de onde veio a geometria: `file`, `file_unverified` ou `template`;
- a tabela de correspondência (e, só com o alvo `"linked_ears"`, os eletrodos substitutos de mastoide);
- a figura: Fp1/Fp2 junto ao nariz, O1/O2 atrás, ímpares à esquerda, pares à direita.

In [ ]:
first_per_site = ok.groupby("site").head(1)
fig, axes = plt.subplots(1, len(first_per_site), figsize=(7.5 * len(first_per_site), 7.5))
axes = np.atleast_1d(axes)
recording_info = {}

for ax, r in zip(axes, first_per_site.itertuples()):
    print("=" * 78)
    print(f"{r.site} | {r.sid} | {Path(r.eeg_file).name}")
    try:
        raw = brainlat_transfer.read_raw_any(r.eeg_file)
    except brainlat_transfer.EpochedFileError as exc:
        print("  ", exc)
        ax.axis("off")
        continue

    print("ESTADO DO ARQUIVO")
    recording_info[r.site] = brainlat_transfer.describe_recording(raw)
    print(f"  primeiros nomes: {raw.ch_names[:10]}")
    print(f"  últimos nomes  : {raw.ch_names[-8:]}")

    print("ADAPTAÇÃO")
    pos, mp, src, notes = channel_adaptation.resolve_geometry(raw, "auto", ORDER)
    print(f"  geometria: {src}")
    for n in notes:
        print("   -", n)
    print(mp.to_string(index=False))
    target = feature_sets.TARGET_REFERENCE
    ref_plot = []
    if target == "linked_ears":
        refs = channel_adaptation.nearest_channels(pos, exclude=set(mp["source"]))
        print("  substitutos de mastoide:")
        print(refs.to_string(index=False))
        ref_plot = refs["source"].tolist()
    else:
        print(f"  referência-alvo '{target}': calculada sobre os 19 canais, sem eletrodos extras")
    channel_adaptation.plot_mapping(pos, mp, ref_plot,
                               title=f"{r.site} ({src}): mediana {mp['angle_deg'].median():.1f}°", ax=ax)

fig.savefig(OUT_DIR / "adaptacao_canais_por_sitio.png", dpi=150, bbox_inches="tight")
plt.show()

Parâmetros da adaptação. A referência-alvo vem de `feature_sets.TARGET_REFERENCE`, a mesma
aplicada ao ds004504 (padrão: média dos 19 canais, que independe da referência de gravação).
Os eletrodos substitutos de mastoide mostrados no passo 2 só são usados se o alvo for
`"linked_ears"`; com canais externos nas mastoides (EXG), passe-os em `ref_channels`.

In [ ]:
ADAPTER_KWARGS = dict(
    mapping_source="auto",        # "auto" | "file" | "template"
    reference=None,               # None = feature_sets.TARGET_REFERENCE | "average" | "rest" | "linked_ears"
    ref_channels=None,            # só para "linked_ears": p.ex. ["EXG1", "EXG2"]
    order=ORDER,
    max_duration_s=None,          # o treino usou o registro inteiro
)
print("referência-alvo:", ADAPTER_KWARGS["reference"] or feature_sets.TARGET_REFERENCE)

## 3 - um sujeito do início ao fim

In [ ]:
sid0 = first_per_site.iloc[0]["sid"]
raw = brainlat_transfer.read_raw_any(paths[sid0])
raw19, prov = channel_adaptation.adapt_to_training_space(raw, verbose=True, **ADAPTER_KWARGS)
assert raw19.ch_names == ORDER and raw19.info["sfreq"] == 500.0

raw_clean, rep = ds004504_features.clean_like_training(raw19)
print(f"ICA: {rep['n_ica_components']} componentes, excluídos {rep['ic_excluded']}")
print("ICLabel:", rep["ic_labels"])

short = raw_clean.copy().crop(tmax=12.0)
pd.DataFrame(ds004504_features.extract_epoch_features(short, sid0)).drop(
    columns=["Subject_ID", "Epoch_ID"]).T

## 4 - todos os sujeitos (com cache)

Cada sujeito gera `brainlat_features/v<versão>/<id>_epochs.csv` e `<id>_provenance.json`. Ao rodar de
novo, só quem falta é processado; `overwrite=True` refaz tudo.

In [ ]:
epochs_df, failures = brainlat_transfer.run(paths, CACHE_DIR, adapter_kwargs=ADAPTER_KWARGS)
print(f"\n{epochs_df['Subject_ID'].nunique()} sujeitos, {len(epochs_df)} épocas")
for k, v in failures.items():
    print("FALHA", k, "->", v)

# Fluxo da amostra: rotulados -> processados, com o motivo de cada exclusão
fluxo = brainlat_transfer.cohort_flow(cohort, failures)
fluxo.to_csv(OUT_DIR / "fluxo_amostra_brainlat.csv", index=False)
print()
print(fluxo.to_string(index=False))

## 5 - a adaptação foi a mesma para todos?

Se a geometria vem do modelo padrão, todos recebem os mesmos eletrodos. Se vem do arquivo,
pequenas variações são possíveis. Deslocamentos discrepantes ou muitos canais interpolados
indicam um sujeito para inspecionar.

In [ ]:
summary = brainlat_transfer.mapping_summary(CACHE_DIR)
summary["site"] = summary["sid"].map(sites)
summary.to_csv(OUT_DIR / "adaptacao_por_sujeito.csv", index=False)

print(summary.groupby(["site", "geometry"]).size().to_string())
print("\nconjuntos de eletrodos distintos:", summary["electrodes"].nunique())
print(summary.groupby("electrodes")["sid"].count().to_string())
print("\nreferências usadas:", summary["ref"].value_counts().to_dict())
print("taxas originais:", summary["orig_sfreq"].value_counts().to_dict())
print("passa-baixa no arquivo:", summary["lowpass_hz"].value_counts().to_dict())
print("arquivo já em referência média:", summary["avg_ref_in_file"].value_counts().to_dict())
print("sujeitos com trechos removidos (boundary):", int((summary["n_boundary"] > 0).sum()))
summary[["sid", "site", "geometry", "median_deg", "max_deg", "interpolated",
         "n_ic_excluded", "n_epochs"]].sort_values("max_deg", ascending=False).head(10)

## 6 - modelos finais (treinados com os 65 sujeitos do ds004504)

In [ ]:
bundle = joblib.load(MODEL_BUNDLE) if MODEL_BUNDLE.exists() else None
if bundle is None or bundle["metadata"].get("feature_tag") != FEATURE_TAG:
    print("treinando os modelos finais (bundle ausente ou de outra versão/referência)")
    bundle = train_final_models.export_models(TRAIN_FEATURES, MODEL_BUNDLE)
print(json.dumps({k: v for k, v in bundle["metadata"].items() if k != "bands"}, indent=2, ensure_ascii=False))

## 7 - classificação

In [ ]:
subj = brainlat_transfer.subject_level(epochs_df)
preds = brainlat_transfer.predict_all(bundle, subj)
preds["site"] = preds["Subject_ID"].map(sites)
preds.to_csv(OUT_DIR / "predicoes.csv", index=False)

metrics = brainlat_transfer.evaluate(preds, labels)
metrics.to_csv(OUT_DIR / "metricas_direto.csv", index=False)
cols = ["Modelo", "Categoria", "n", "Accuracy", "Acc_IC_inf", "Acc_IC_sup", "AUC", "AUC_IC_inf",
        "AUC_IC_sup", "Sensibilidade", "Especificidade", "Frac_prevista_AD", "Classe_majoritaria"]
metrics[cols].sort_values("AUC", ascending=False).round(3)

In [ ]:
# Por sítio (AR: amostra completa; CL: só 13 de 27 controles, ver fluxo da amostra)
por_sitio = pd.concat([brainlat_transfer.evaluate(g, labels).assign(site=s)
                       for s, g in preds.groupby("site")])
por_sitio.to_csv(OUT_DIR / "metricas_por_sitio.csv", index=False)
por_sitio.pivot_table(index=["Modelo", "Categoria"], columns="site",
                      values=["n", "AUC", "AUC_IC_inf", "AUC_IC_sup"]).round(3)

## 7b - referência: os mesmos modelos treinados dentro do BrainLat

Mesmos pipelines e mesma validação LOSO, mas treinados e avaliados só com sujeitos do BrainLat.
Se o desempenho interno também for baixo, a queda da transferência não vem só do
deslocamento entre bases: o problema pode ser mais difícil nesta coorte.

In [ ]:
interno, interno_preds = brainlat_transfer.within_cohort_loso(subj, labels, verbose=True)
interno.to_csv(OUT_DIR / "metricas_loso_interno_brainlat.csv", index=False)

comp = metrics[["Modelo", "Categoria", "AUC"]].merge(
    interno[["Modelo", "Categoria", "AUC", "AUC_IC_inf", "AUC_IC_sup"]],
    on=["Modelo", "Categoria"], suffixes=("_transferido", "_interno"))
comp.sort_values("AUC_interno", ascending=False).round(3)

## 8 - deslocamento de domínio

`d_HC_shift`: controles BrainLat vs controles ds004504. `d_AD_train`: efeito AD-vs-CN aprendido.
`d_AD_target`: efeito AD-vs-CN no BrainLat. `same_sign = False` numa feature importante, ou
`|d_HC_shift|` da ordem de `|d_AD_train|`, impede a transferência sem harmonização.

In [ ]:
shift = brainlat_transfer.domain_shift_report(bundle["train_subject_features"], subj, labels)
shift.to_csv(OUT_DIR / "deslocamento_dominio.csv", index=False)
shift.sort_values("d_HC_shift", key=np.abs, ascending=False)

## 9 - (opcional) alinhamento pelos controles

Metade dos controles de cada sítio (sorteio fixo) calibra média e desvio de cada feature
contra os controles do ds004504 e sai da avaliação. Reporte separado do resultado direto.

In [ ]:
rng = np.random.default_rng(42)
calib_ids = []
for s in sorted(sites.unique()):
    hc_s = sorted(labels[(labels == 0) & (sites == s)].index)
    calib_ids += list(rng.choice(hc_s, size=len(hc_s) // 2, replace=False))
print(f"controles de calibração: {len(calib_ids)}")

subj_al = brainlat_transfer.align_to_training_controls(subj, bundle["train_subject_features"], calib_ids)
subj_eval = subj_al[~subj_al["Subject_ID"].isin(set(calib_ids))]
metrics_al = brainlat_transfer.evaluate(brainlat_transfer.predict_all(bundle, subj_eval), labels)
metrics_al.to_csv(OUT_DIR / "metricas_alinhado.csv", index=False)
with open(OUT_DIR / "controles_calibracao.json", "w", encoding="utf-8") as fh:
    json.dump(sorted(calib_ids), fh, indent=2)
metrics_al.sort_values(["AUC", "Accuracy"], ascending=False)